<a href="https://colab.research.google.com/github/BigFoots625/IntroductionMachineLearningwithPython/blob/main/Chapter7_Working_with_TextData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Working with Text Data**
In this chapter, we will explore Natural Language Processing (NLP) techniques for machine learning.

Before diving into algorithms, we must understand that text data typically comes in four forms:
1. **Categorical Data:** Text that belongs to a fixed list of options (e.g., "Red", "Green", "Blue"). We handle this with One-Hot Encoding, as covered in Chapter 4.
2. **Free Strings that can be semantically mapped:** Text like "John Smith" or "London". These require specialized parsing to extract meaning (e.g., recognizing that "London" is a city).
3. **Structured String Data:** Text with a specific format, like URLs, email addresses, or phone numbers.
4. **Text Data / Natural Language:** Free-form text consisting of phrases or sentences, like a movie review or a tweet. This is the focus of this chapter.

In [1]:
!pip install mglearn spacy
!python -m spacy download en_core_web_sm
import mglearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.datasets import fetch_20newsgroups
import warnings
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.4/581.4 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### **Theoretical Explanation: Bag-of-Words (BoW)**
The most common way to represent text in machine learning is the **Bag-of-Words** approach. It completely ignores the structure, grammar, and order of words in a sentence, and simply counts how often each word appears.

Computing the Bag-of-Words representation consists of three steps:
1. **Tokenization:** Splitting each document (e.g., a sentence) into the words that appear in it (tokens), usually by splitting on whitespace and punctuation.
2. **Vocabulary Building:** Collecting a vocabulary of all words that appear in any of the documents, and numbering them (e.g., in alphabetical order).
3. **Encoding:** For each document, counting how often each word in the vocabulary appears in that document.

Because most documents only contain a small subset of the total vocabulary, the resulting matrix is mostly filled with zeros. Scikit-learn stores this as a **SciPy sparse matrix** to save memory.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

# A simple toy dataset of two documents
bards_words = ["The fool doth think he is wise,",
               "but the wise man knows himself to be a fool"]

# Instantiate the vectorizer and fit it to build the vocabulary
vect = CountVectorizer()
vect.fit(bards_words)

print("Vocabulary size:", len(vect.vocabulary_))
print("Vocabulary content:\n", vect.vocabulary_)

# Transform the documents into the sparse matrix representation
bag_of_words = vect.transform(bards_words)
print("\nDense representation of the Bag-of-Words:\n", bag_of_words.toarray())

Vocabulary size: 13
Vocabulary content:
 {'the': 9, 'fool': 3, 'doth': 2, 'think': 10, 'he': 4, 'is': 6, 'wise': 12, 'but': 1, 'man': 8, 'knows': 7, 'himself': 5, 'to': 11, 'be': 0}

Dense representation of the Bag-of-Words:
 [[0 0 1 1 1 0 1 0 0 1 1 0 1]
 [1 1 0 1 0 1 0 1 1 1 0 1 1]]


### **Theoretical Explanation: Stopwords**
When we build a vocabulary across a large corpus of text, we will find that some words are incredibly common (like "the", "a", "is", "and"). These words appear in almost every document and therefore carry very little semantic meaning or predictive power.

There are two ways to get rid of them:
1. **Stopword Lists:** Using a built-in language-specific list of words to automatically drop. Scikit-learn has a built-in `english` stopword list.
2. **Frequency Thresholds:** Discarding words that appear too frequently across documents (using the `max_df` parameter) or words that appear too rarely (using the `min_df` parameter, which helps eliminate typos and extremely rare terms).

In [3]:
# Using English stopwords and a minimum document frequency of 2
vect_stopwords = CountVectorizer(stop_words="english")
vect_stopwords.fit(bards_words)

print("Vocabulary with stopwords removed:\n", vect_stopwords.vocabulary_)

# Note how words like "the", "he", "is", "to", "be", "a" were dropped.

Vocabulary with stopwords removed:
 {'fool': 1, 'doth': 0, 'think': 4, 'wise': 5, 'man': 3, 'knows': 2}


### **Theoretical Explanation: TF-IDF (Term Frequency - Inverse Document Frequency)**
Instead of simply dropping common words, a more mathematically sound approach is to rescale features based on how informative they are. We want to give high weight to terms that appear frequently in a *specific* document, but rarely across the *entire* corpus.

This is exactly what TF-IDF does. Scikit-learn implements it using the following formula:
$$tfidf(w, d) = tf(w, d) \times \log\left(\frac{N + 1}{N_w + 1}\right) + 1$$

Where:
* $tf(w, d)$ is the Term Frequency (how often word $w$ appears in document $d$).
* $N$ is the total number of documents in the training set.
* $N_w$ is the number of documents in the training set that contain the word $w$.

After computing this, `TfidfVectorizer` applies L2 normalization (like the `Normalizer` we learned in Chapter 3), which means every document vector is scaled to have a Euclidean length of 1.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer()
tfidf_vect.fit(bards_words)

# Transform the text and look at the TF-IDF weights
tfidf_matrix = tfidf_vect.transform(bards_words)
print("TF-IDF matrix (dense):\n", tfidf_matrix.toarray())

TF-IDF matrix (dense):
 [[0.         0.         0.42567716 0.30287281 0.42567716 0.
  0.42567716 0.         0.         0.30287281 0.42567716 0.
  0.30287281]
 [0.36469323 0.36469323 0.         0.25948224 0.         0.36469323
  0.         0.36469323 0.36469323 0.25948224 0.         0.36469323
  0.25948224]]


### **Theoretical Explanation: n-Grams**
The major flaw of the Bag-of-Words model is that word order is completely discarded. The sentence *"it's bad, not good at all"* and *"it's good, not bad at all"* have the exact same Bag-of-Words representation, despite having opposite meanings.

To capture context, we can extract **n-grams**—sequences of $n$ adjacent words.
* **Unigrams (n=1):** Single words (standard Bag-of-Words).
* **Bigrams (n=2):** Pairs of adjacent words (e.g., "not good", "good not").
* **Trigrams (n=3):** Triplets of adjacent words.

Using bigrams allows the model to capture negation ("not bad" vs. "not good"), significantly boosting the performance of text classifiers. We control this using the `ngram_range` parameter.

In [5]:
# Extract unigrams AND bigrams by setting ngram_range=(1, 2)
ngram_vect = CountVectorizer(ngram_range=(1, 2))
ngram_vect.fit(bards_words)

print("Vocabulary size with unigrams and bigrams:", len(ngram_vect.vocabulary_))
print("Sample of the vocabulary:\n", list(ngram_vect.vocabulary_.keys())[:10])

Vocabulary size with unigrams and bigrams: 27
Sample of the vocabulary:
 ['the', 'fool', 'doth', 'think', 'he', 'is', 'wise', 'the fool', 'fool doth', 'doth think']


### **Theoretical Explanation: Stemming and Lemmatization**
In natural language, the same word can take many forms (e.g., "draw", "draws", "drawing", "drew"). If we use a standard `CountVectorizer`, these are treated as four completely separate features, diluting the model's ability to learn that they relate to the same concept.

We can normalize the text to merge these variations:
1. **Stemming:** A crude, rule-based approach that simply chops off word endings (e.g., dropping "ing" or "s"). It is fast but can produce non-words.
2. **Lemmatization:** A complex, dictionary-based approach that understands grammar and converts a word to its known dictionary root (the lemma). For example, it knows that "was" is a form of "be".

Scikit-learn does not have built-in lemmatization, so we must use external NLP libraries like `spacy` or `nltk` to process the text before passing it to the vectorizer.

In [6]:
import spacy

# Load the English language model
en_nlp = spacy.load('en_core_web_sm')

# A function to lemmatize a string
def custom_tokenizer(document):
    # Parse the document using spacy
    doc_spacy = en_nlp(document)
    # Return the lemma for each token, ignoring punctuation and spaces
    return [token.lemma_ for token in doc_spacy if not token.is_punct and not token.is_space]

# Pass our custom tokenizer into the CountVectorizer
lemma_vect = CountVectorizer(tokenizer=custom_tokenizer)
lemma_vect.fit(["I am drawing", "He drew", "She will draw"])

print("Lemmatized vocabulary (notice how they all map to 'draw' and 'be'):")
print(lemma_vect.vocabulary_)

Lemmatized vocabulary (notice how they all map to 'draw' and 'be'):
{'I': 0, 'be': 1, 'draw': 2, 'he': 3, 'she': 4, 'will': 5}


### **Theoretical Explanation: Topic Modeling and LDA**
Topic modeling is an **unsupervised learning** technique for text. We don't have labels (like "positive review" or "spam"); we just want the algorithm to discover the underlying topics discussed in a massive corpus of documents.

The most prominent algorithm for this is **Latent Dirichlet Allocation (LDA)**.
LDA operates on the assumption that:
1. Every document is a mixture of various topics.
2. Every topic is a mixture of various words.

For example, a news article about a politician proposing an environmental law might be 50% about the topic "Politics" and 50% about "Environment". LDA analyzes the co-occurrence of words across all documents to simultaneously discover what the topics are (by finding words that group together) and which topics belong to which documents.

In [7]:
# We will use a small sample of the 20 Newsgroups dataset for this demonstration
newsgroups = fetch_20newsgroups(subset='train', categories=['sci.space', 'rec.autos'])
text_data = newsgroups.data[:500] # Use 500 documents for speed

# 1. Vectorize the data (LDA requires raw word counts, so we use CountVectorizer, NOT TF-IDF)
# We remove common words and words that appear in more than 15% of documents
lda_vect = CountVectorizer(max_features=1000, max_df=0.15, stop_words="english")
X_topics = lda_vect.fit_transform(text_data)

# 2. Fit the LDA model, asking it to find 2 underlying topics
lda = LatentDirichletAllocation(n_components=2, learning_method="batch", max_iter=10, random_state=0)
document_topics = lda.fit_transform(X_topics)

# 3. Print the top words for each discovered topic
feature_names = np.array(lda_vect.get_feature_names_out())
for topic_idx, topic in enumerate(lda.components_):
    print(f"\nTopic {topic_idx + 1}:")
    # Sort the words by their weight in the topic and print the top 5
    top_words = feature_names[np.argsort(topic)[-5:]]
    print(top_words)


Topic 1:
['data' 'moon' 'earth' 'orbit' 'gov']

Topic 2:
['henry' 'engine' 'year' 'cars' 'launch']


### **Chapter 7 Summary**
* **Objective:** We learned how to extract numerical features from unstructured text data to use in machine learning pipelines.
* **Key Concepts Covered:**
    * **Bag-of-Words:** Representing text as a sparse matrix of word counts (`CountVectorizer`).
    * **Stopwords & TF-IDF:** Filtering out noise and mathematically weighting terms based on their rarity across the corpus (`TfidfVectorizer`).
    * **n-Grams:** Capturing local word context and negations by extracting overlapping sequences of words.
    * **Lemmatization:** Standardizing text to its dictionary roots using tools like `spaCy` to reduce vocabulary dilution.
    * **Topic Modeling:** Using Latent Dirichlet Allocation (LDA) to discover hidden thematic structures in large text datasets without supervision.
* **Takeaway:** Text processing requires careful consideration. A pipeline using `TfidfVectorizer` with bigrams, fed into a simple Logistic Regression, is often the most robust baseline for text classification tasks.